<a href="https://colab.research.google.com/github/gloriabee/DEMovies/blob/main/silver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import duckdb

In [ ]:
DRIVE_DB= "/content/drive/MyDrive/data_engineering/movies.duckdb"
LOCAL_DB="/content/movies.duckdb"

In [ ]:
print("Exists:",os.path.exists(DRIVE_DB))

In [ ]:
if os.path.exists(DRIVE_DB):
  print("Drive DB size:",os.path.getsize(DRIVE_DB))


In [ ]:
con=duckdb.connect(DRIVE_DB)

In [ ]:
con.execute("SHOW TABLES").fetchdf()

In [ ]:
movies_df=con.execute("""
SELECT *
FROM movies_bronze
""").fetchdf()

In [ ]:
movies_df.head()

## Start Silver Transformations

### remove irrelevant columns

In [ ]:
movies_df.columns

In [ ]:
movies_df=movies_df.drop(columns=['IMDbLink','Title_IMDb_Link'])

In [ ]:
movies_df.head()

### standardization of column names

In [ ]:
movies_df.info()

In [ ]:
for col in ["directors","writers","genres"]:
  print(col, (movies_df[col].str.strip()=="").sum())

In [ ]:
print("Duplicate tconst:",movies_df["tconst"].duplicated().sum())

In [ ]:
invalid_rating=movies_df[
    (movies_df['averageRating']>10) |
    (movies_df['averageRating']<0)
]

print("Invalid rating :",len(invalid_rating))

In [ ]:
invalid_runtime=movies_df[
    movies_df['runtimeMinutes']<=0
]

print("Invalid rows:",len(invalid_runtime))

In [ ]:
invalid_year=movies_df[
    (movies_df['startYear']>2026) |
    (movies_df['startYear']<1900)
]
print("Invalid rows: ",len(invalid_year))
